### Import

In [1]:
import pandas as pd
import numpy as np
import gc

import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

### Data Load

In [3]:
# 데이터 분할(폴더) 구분
data_splits = ["train", "test"]

# 각 데이터 유형별 폴더명, 파일 접미사, 변수 접두어 설정
data_categories = {
    "회원정보": {"folder": "1.회원정보", "suffix": "회원정보", "var_prefix": "customer"},
    "신용정보": {"folder": "2.신용정보", "suffix": "신용정보", "var_prefix": "credit"},
    "승인매출정보": {"folder": "3.승인매출정보", "suffix": "승인매출정보", "var_prefix": "sales"},
    "청구정보": {"folder": "4.청구입금정보", "suffix": "청구정보", "var_prefix": "billing"},
    "잔액정보": {"folder": "5.잔액정보", "suffix": "잔액정보", "var_prefix": "balance"},
    "채널정보": {"folder": "6.채널정보", "suffix": "채널정보", "var_prefix": "channel"},
    "마케팅정보": {"folder": "7.마케팅정보", "suffix": "마케팅정보", "var_prefix": "marketing"},
    "성과정보": {"folder": "8.성과정보", "suffix": "성과정보", "var_prefix": "performance"}
}

# 2018년 7월부터 12월까지의 월 리스트
months = ['07', '08', '09', '10', '11', '12']

for split in data_splits:
    for category, info in data_categories.items():
        folder = info["folder"]
        suffix = info["suffix"]
        var_prefix = info["var_prefix"]
        
        for month in months:
            # 파일명 형식: 2018{month}_{split}_{suffix}.parquet
            file_path = f"./datasets/{split}/{folder}/2018{month}_{split}_{suffix}.parquet"
            # 변수명 형식: {var_prefix}_{split}_{month}
            variable_name = f"{var_prefix}_{split}_{month}"
            globals()[variable_name] = pd.read_parquet(file_path)
            print(f"{variable_name} is loaded from {file_path}")

gc.collect()

customer_train_07 is loaded from ./datasets/train/1.회원정보/201807_train_회원정보.parquet
customer_train_08 is loaded from ./datasets/train/1.회원정보/201808_train_회원정보.parquet
customer_train_09 is loaded from ./datasets/train/1.회원정보/201809_train_회원정보.parquet
customer_train_10 is loaded from ./datasets/train/1.회원정보/201810_train_회원정보.parquet
customer_train_11 is loaded from ./datasets/train/1.회원정보/201811_train_회원정보.parquet
customer_train_12 is loaded from ./datasets/train/1.회원정보/201812_train_회원정보.parquet
credit_train_07 is loaded from ./datasets/train/2.신용정보/201807_train_신용정보.parquet
credit_train_08 is loaded from ./datasets/train/2.신용정보/201808_train_신용정보.parquet
credit_train_09 is loaded from ./datasets/train/2.신용정보/201809_train_신용정보.parquet
credit_train_10 is loaded from ./datasets/train/2.신용정보/201810_train_신용정보.parquet
credit_train_11 is loaded from ./datasets/train/2.신용정보/201811_train_신용정보.parquet
credit_train_12 is loaded from ./datasets/train/2.신용정보/201812_train_신용정보.parquet
sales_train_07 i

0

### Data Preprocessing(1) : Concat & Merge

In [4]:
# 데이터 유형별 설정 
info_categories = ["customer", "credit", "sales", "billing", "balance", "channel", "marketing", "performance"]

# 월 설정
months = ['07', '08', '09', '10', '11', '12']

In [5]:
#### Train ####

# 각 유형별로 월별 데이터를 합쳐서 새로운 변수에 저장
train_dfs = {}

for prefix in info_categories:
    # globals()에서 동적 변수명으로 데이터프레임들을 가져와 리스트에 저장
    df_list = [globals()[f"{prefix}_train_{month}"] for month in months]
    train_dfs[f"{prefix}_train_df"] = pd.concat(df_list, axis=0)
    gc.collect()
    print(f"{prefix}_train_df is created with shape: {train_dfs[f'{prefix}_train_df'].shape}")


customer_train_df = train_dfs["customer_train_df"]
credit_train_df   = train_dfs["credit_train_df"]
sales_train_df    = train_dfs["sales_train_df"]
billing_train_df  = train_dfs["billing_train_df"]
balance_train_df  = train_dfs["balance_train_df"]
channel_train_df  = train_dfs["channel_train_df"]
marketing_train_df= train_dfs["marketing_train_df"]
performance_train_df = train_dfs["performance_train_df"]

gc.collect()

customer_train_df is created with shape: (2400000, 78)
credit_train_df is created with shape: (2400000, 42)
sales_train_df is created with shape: (2400000, 406)
billing_train_df is created with shape: (2400000, 46)
balance_train_df is created with shape: (2400000, 82)
channel_train_df is created with shape: (2400000, 105)
marketing_train_df is created with shape: (2400000, 64)
performance_train_df is created with shape: (2400000, 49)


0

In [6]:
#### Test ####

# test 데이터에 대해 train과 동일한 방법 적용
test_dfs = {}

for prefix in info_categories:
    df_list = [globals()[f"{prefix}_test_{month}"] for month in months]
    test_dfs[f"{prefix}_test_df"] = pd.concat(df_list, axis=0)
    gc.collect()
    print(f"{prefix}_test_df is created with shape: {test_dfs[f'{prefix}_test_df'].shape}")


customer_test_df = test_dfs["customer_test_df"]
credit_test_df   = test_dfs["credit_test_df"]
sales_test_df    = test_dfs["sales_test_df"]
billing_test_df  = test_dfs["billing_test_df"]
balance_test_df  = test_dfs["balance_test_df"]
channel_test_df  = test_dfs["channel_test_df"]
marketing_test_df= test_dfs["marketing_test_df"]
performance_test_df = test_dfs["performance_test_df"]

gc.collect()

customer_test_df is created with shape: (600000, 77)
credit_test_df is created with shape: (600000, 42)
sales_test_df is created with shape: (600000, 406)
billing_test_df is created with shape: (600000, 46)
balance_test_df is created with shape: (600000, 82)
channel_test_df is created with shape: (600000, 105)
marketing_test_df is created with shape: (600000, 64)
performance_test_df is created with shape: (600000, 49)


0

In [7]:
#### Train ####

train_df = customer_train_df.merge(credit_train_df, on=['기준년월', 'ID'], how='left')
print("Step1 저장 완료: train_step1, shape:", train_df.shape)
del customer_train_df, credit_train_df
gc.collect()

# 이후 merge할 데이터프레임 이름과 단계 정보를 리스트에 저장
merge_list = [
    ("sales_train_df",    "Step2"),
    ("billing_train_df",  "Step3"),
    ("balance_train_df",  "Step4"),
    ("channel_train_df",  "Step5"),
    ("marketing_train_df","Step6"),
    ("performance_train_df", "최종")
]

# 나머지 단계 merge
for df_name, step in merge_list:
    # globals()로 동적 변수 접근하여 merge 수행
    train_df = train_df.merge(globals()[df_name], on=['기준년월', 'ID'], how='left')
    print(f"{step} 저장 완료: train_{step}, shape:", train_df.shape)
    # 사용한 변수는 메모리 해제를 위해 삭제
    del globals()[df_name]
    gc.collect()

Step1 저장 완료: train_step1, shape: (2400000, 118)
Step2 저장 완료: train_Step2, shape: (2400000, 522)
Step3 저장 완료: train_Step3, shape: (2400000, 566)
Step4 저장 완료: train_Step4, shape: (2400000, 646)
Step5 저장 완료: train_Step5, shape: (2400000, 749)
Step6 저장 완료: train_Step6, shape: (2400000, 811)
최종 저장 완료: train_최종, shape: (2400000, 858)


In [8]:
#### Test ####

test_df = customer_test_df.merge(credit_test_df, on=['기준년월', 'ID'], how='left')
print("Step1 저장 완료: test_step1, shape:", test_df.shape)
del customer_test_df, credit_test_df
gc.collect()

# 이후 merge할 데이터프레임 이름과 단계 정보를 리스트에 저장
merge_list = [
    ("sales_test_df",    "Step2"),
    ("billing_test_df",  "Step3"),
    ("balance_test_df",  "Step4"),
    ("channel_test_df",  "Step5"),
    ("marketing_test_df","Step6"),
    ("performance_test_df", "최종")
]

# 나머지 단계 merge
for df_name, step in merge_list:
    # globals()로 동적 변수 접근하여 merge 수행
    test_df = test_df.merge(globals()[df_name], on=['기준년월', 'ID'], how='left')
    print(f"{step} 저장 완료: test_{step}, shape:", test_df.shape)
    # 사용한 변수는 메모리 해제를 위해 삭제
    del globals()[df_name]
    gc.collect()

Step1 저장 완료: test_step1, shape: (600000, 117)
Step2 저장 완료: test_Step2, shape: (600000, 521)
Step3 저장 완료: test_Step3, shape: (600000, 565)
Step4 저장 완료: test_Step4, shape: (600000, 645)
Step5 저장 완료: test_Step5, shape: (600000, 748)
Step6 저장 완료: test_Step6, shape: (600000, 810)
최종 저장 완료: test_최종, shape: (600000, 857)


### Data Preprocessing(2) : Encoding

In [9]:
feature_cols = [col for col in train_df.columns if col not in ["ID", "Segment"]]

X = train_df[feature_cols].copy()
y = train_df["Segment"].copy()

# 타깃 라벨 인코딩
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

In [10]:
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

X_test = test_df.copy()

encoders = {}  # 각 컬럼별 encoder 저장

for col in categorical_features:
    le_train = LabelEncoder()
    X[col] = le_train.fit_transform(X[col])
    encoders[col] = le_train
    unseen_labels_val = set(X_test[col]) - set(le_train.classes_)
    if unseen_labels_val:
        le_train.classes_ = np.append(le_train.classes_, list(unseen_labels_val))
    X_test[col] = le_train.transform(X_test[col])

In [11]:
gc.collect()

0

In [16]:
X

,기준년월,남녀구분코드,연령,회원여부_이용가능,회원여부_이용가능_CA,회원여부_이용가능_카드론,소지여부_신용,소지카드수_유효_신용,소지카드수_이용가능_신용,입회일자_신용,...,변동률_RV일시불평잔,변동률_할부평잔,변동률_CA평잔,변동률_RVCA평잔,변동률_카드론평잔,변동률_잔액_B1M,변동률_잔액_일시불_B1M,변동률_잔액_CA_B1M,혜택수혜율_R3M,혜택수혜율_B0M
0,201807,2,2,1,1,0,1,1,1,20130101,...,0.999998,1.042805,0.999700,0.999998,0.999998,0.261886,0.270752,0.000000,1.044401,1.280543
1,201807,1,1,1,1,1,1,1,1,20170801,...,1.092698,0.905663,0.999998,0.999998,0.999998,-0.563388,-0.670348,0.000000,0.000000,0.000000
2,201807,1,1,1,1,0,1,1,1,20080401,...,1.006124,1.993590,0.852567,0.999998,0.999998,-0.046516,0.058114,-0.014191,0.524159,1.208420
3,201807,2,2,1,1,0,1,2,2,20160501,...,0.999998,1.050646,0.999877,0.999998,0.999998,0.023821,0.258943,0.000000,0.880925,1.657124
4,201807,2,2,1,1,1,1,1,1,20180601,...,0.999998,0.999998,0.999998,0.999998,0.999998,0.000000,0.000000,0.000000,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,201812,2,5,1,1,1,1,1,1,20010701,...,0.999998,0.999998,0.999998,0.999998,0.999998,0.000000,0.000000,0.000000,NaN,NaN
2399996,201812,2,3,1,1,1,1,1,1,20170701,...,0.999998,0.999998,0.999998,0.999998,0.921733,-0.203251,-0.159143,0.000000,1.377071,2.533815
2399997,201812,1,1,1,1,0,1,1,1,20090501,...,0.999998,0.345027,0.999998,0.999998,0.999998,0.027319,0.126581,0.000000,0.000000,0.000000
2399998,201812,1,2,1,1,1,1,1,1,20130101,...,0.999998,0.999998,0.999998,0.999998,0.999998,0.000000,0.000000,0.000000,NaN,NaN


### Train

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np
import time
import multiprocessing
import random
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# 재현성 보장을 위한 시드 설정
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 시드 설정
#set_seed(42)

# MPS 디바이스 확인
print(f"MPS 사용 가능: {torch.backends.mps.is_available()}")
print(f"MPS 사용 중: {torch.backends.mps.is_built()}")
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"사용 중인 디바이스: {device}")

# CPU 코어 수 확인 및 워커 수 제한
num_workers = min(6, multiprocessing.cpu_count())
print(f"사용 가능한 CPU 코어 수: {multiprocessing.cpu_count()}")
print(f"실제 사용할 워커 수: {num_workers}")

# 커스텀 데이터셋 정의
class CardDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

def dataLoadAndPreprocess(train_df=None, test_df=None, label_df=None):
    
    if train_df is None:
        # 데이터 로드 (CPU에서 수행)
        train_df = pd.read_csv('train_pca_df.csv').values
        test_df = pd.read_csv('test_pca_df.csv').values
        label_df = pd.read_csv('target_train_df.csv').values.reshape(-1, 1)
    
    
    # 레이블 인코딩
    le = LabelEncoder()
    label_df = le.fit_transform(label_df)
    
    # 스케일링 적용
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_df)
    test_scaled = scaler.transform(test_df)
    
    # train_test_split 적용
    train_data, val_data, train_labels, val_labels = train_test_split(
        train_scaled, label_df, test_size=0.2, random_state=42, stratify=label_df
    )
    
    # PyTorch 텐서로 변환 (CPU에서 수행)
    train_tensor = torch.FloatTensor(train_data)
    val_tensor = torch.FloatTensor(val_data)
    test_tensor = torch.FloatTensor(test_scaled)
    train_label_tensor = torch.LongTensor(train_labels)
    val_label_tensor = torch.LongTensor(val_labels)
    
    return train_tensor, val_tensor, test_tensor, train_label_tensor, val_label_tensor, le

print("데이터 로드 중...")
train_tensor, val_tensor, test_tensor, train_label_tensor, val_label_tensor, le = dataLoadAndPreprocess()
print("데이터 로드 완료")

print(train_tensor.shape)
print(val_tensor.shape)
print(test_tensor.shape)
print(train_label_tensor.shape)
print(val_label_tensor.shape)

# 데이터셋 생성
train_dataset = CardDataset(train_tensor, train_label_tensor)
val_dataset = CardDataset(val_tensor, val_label_tensor)

# DataLoader 생성
train_dataloader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False,
    prefetch_factor=None
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False,
    prefetch_factor=None
)

# 간단한 모델 정의
class SimpleModel(nn.Module):
    def __init__(self, input_dim=812, hidden_dim=512, output_dim=5):
        super(SimpleModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, output_dim)
        self.relu = nn.ReLU()
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(256)
        self.bn3 = nn.BatchNorm1d(128)
        self.bn4 = nn.BatchNorm1d(64)
        self.dropout = nn.Dropout(0.2)
        
    def forward(self, x):
        x = self.dropout(self.relu(self.bn1(self.fc1(x))))
        x = self.dropout(self.relu(self.bn2(self.fc2(x))))
        x = self.dropout(self.relu(self.bn3(self.fc3(x))))
        x = self.dropout(self.relu(self.bn4(self.fc4(x))))
        return self.fc5(x)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
    
    accuracy = 100 * correct / total
    avg_loss = total_loss / len(dataloader)
    return avg_loss, accuracy

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        
    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(reduction='none')(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * ce_loss
        return focal_loss.mean()


MPS 사용 가능: True
MPS 사용 중: True
사용 중인 디바이스: mps
사용 가능한 CPU 코어 수: 10
실제 사용할 워커 수: 6
데이터 로드 중...


/opt/anaconda3/envs/deep-learning/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


데이터 로드 완료
torch.Size([320000, 812])
torch.Size([80000, 812])
torch.Size([100000, 812])
torch.Size([320000])
torch.Size([80000])


In [13]:
# 전역 변수 선언
global test_tensor, le
    
# 모델, 손실 함수, 옵티마이저 설정
print("\n=== 모델 설정 ===")
model = SimpleModel(input_dim=train_tensor.shape[1], output_dim=len(torch.unique(train_label_tensor))).to(device)
    
# 손실 함수 설정
criterion_ce = nn.CrossEntropyLoss()
criterion_focal = FocalLoss(alpha=1, gamma=2)
    
# 손실 함수 가중치 설정
loss_weights = {'ce': 0.3, 'focal': 0.7}
    
optimizer = optim.Adam(model.parameters(), lr=0.0005)


=== 모델 설정 ===


In [14]:
# 학습 루프
print("\n=== 학습 시작 ===")
num_epochs = 20
best_val_acc = 0.0

start_time = time.time()
for epoch in tqdm(range(num_epochs), desc="Epoch"):
    # 학습
    model.train()
    epoch_loss = 0.0
    epoch_acc = 0.0
    batch_count = 0
        
    try:
        for batch_idx, (inputs, targets) in enumerate(train_dataloader):
            try:
                inputs = inputs.to(device)
                targets = targets.to(device)
                    
                outputs = model(inputs)
                    
                # Combined loss 계산
                loss_ce = criterion_ce(outputs, targets)
                loss_focal = criterion_focal(outputs, targets)
                loss = loss_weights['ce'] * loss_ce + loss_weights['focal'] * loss_focal
                    
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
                epoch_acc += (outputs.argmax(dim=1) == targets).float().mean().item()
                batch_count += 1
                                
            except Exception as e:
                print(f"배치 처리 중 오류 발생: {e}")
                continue
        
    except Exception as e:
        print(f"에폭 처리 중 오류 발생: {e}")
        continue
        
    # 검증
    val_loss, val_acc = evaluate(model, val_dataloader, criterion_ce, device)
        
    if batch_count > 0:
        print(f"\nEpoch {epoch+1} 완료:")
        print(f"Train Loss: {epoch_loss/batch_count:.4f}")
        print(f"Train Accuracy: {epoch_acc/batch_count:.4f}")
        print(f"Validation Loss: {val_loss:.4f}")
        print(f"Validation Accuracy: {val_acc:.2f}%")
            
        # 최고 검증 정확도 저장
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            print(f"새로운 최고 검증 정확도: {best_val_acc:.2f}%")
        
    elapsed_time = time.time() - start_time
    print(f"경과 시간: {elapsed_time/60:.2f}분")

print("\n=== 학습 완료 ===")
total_time = time.time() - start_time
print(f"총 학습 시간: {total_time/60:.2f}분")
print(f"최고 검증 정확도: {best_val_acc:.2f}%")


=== 학습 시작 ===


Epoch:   0%|          | 0/20 [00:11<?, ?it/s]


KeyboardInterrupt: 

In [ ]:

# 예측 수행
model.eval()
with torch.no_grad():
    # 테스트 데이터를 MPS 디바이스로 이동
    test_tensor = test_tensor.to(device)
    prediction = model(test_tensor)

# MPS 텐서를 CPU로 이동한 후 NumPy 배열로 변환
prediction = prediction.cpu().numpy()
prediction = prediction.argmax(axis=1)
prediction = le.inverse_transform(prediction)

submission = pd.read_csv('./datasets/sample_submission.csv')
submission['Segment'] = prediction
submission.to_csv('deep-submission.csv', index=False)

In [16]:
try:
    model = xgb.XGBClassifier(
        tree_method='gpu_hist',  # GPU 모드 설정
        gpu_id=0,
        random_state=42
    )
    print("GPU 사용 가능: gpu_hist 모드 적용")
    model.fit(X, y_encoded)
    
except Exception:
    model = xgb.XGBClassifier(
        random_state=42
    )
    print("GPU 사용 불가: CPU 모드 적용")
    model.fit(X, y_encoded)

GPU 사용 가능: gpu_hist 모드 적용


/opt/anaconda3/envs/mldl/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [09:10:22] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1738880277541/work/src/common/error_msg.cc:45: `gpu_id` is deprecated since2.0.0, use `device` instead. E.g. device=cpu/cuda/cuda:0
  warnings.warn(smsg, UserWarning)
/opt/anaconda3/envs/mldl/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [09:10:22] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1738880277541/work/src/context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)


GPU 사용 불가: CPU 모드 적용


### Predict

In [17]:
X_test.drop(columns=['ID'],inplace=True)

In [18]:
# row-level 예측 수행
y_test_pred = model.predict(X_test)
# 예측 결과를 변환
y_test_pred_labels = le_target.inverse_transform(y_test_pred)

# row 단위 예측 결과를 test_data에 추가
test_data = test_df.copy()  # 원본 유지
test_data["pred_label"] = y_test_pred_labels

### Submission

In [19]:
submission = test_data.groupby("ID")["pred_label"] \
    .agg(lambda x: x.value_counts().idxmax()) \
    .reset_index()

submission.columns = ["ID", "Segment"]

In [20]:
submission.to_csv('./base_submit.csv',index=False)